In [11]:
#Loading in Packages and Data

#Importing Packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time
import pickle
import h5py
###############################################################
def coefs(coefficients,degree):
    coef=coefficients
    coefs=""
    for n in range(degree, -1, -1):
        string=f"({coefficients[len(coef)-(n+1)]:.1e})"
        coefs+=string + f"x^{n}"
        if n != 0:
            coefs+=" + "
    return coefs
###############################################################

# #Importing Model Data
    
# dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
# data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_test7tundra-7_062217.nc') #***
# true_time=data['time']
# # parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_test5tundra-7_062217.nc') #***
# times=data['time'].values/(1e9 * 60); times=times.astype(float);
# Np_str='125e3'
# #Restricts the timesteps of the data from timesteps0 to 140
# data=data.isel(time=np.arange(0,140+1))
# # parcel=parcel.isel(time=np.arange(0,140+1))
# res='1km'

dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_1e6.nc') #***
true_time=data['time']
parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_1e6.nc') #***
times=data['time'].values/(1e9 * 60); times=times.astype(float);
Np_str='1e6'
#Restricts the timesteps of the data from timesteps0 to 140
res='1km'
job_array=False;index_adjust=0
ocean_fraction=0.25


# #uncomment if using 250m data
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_250m.nc') #***
# # # parcel=xr.open_dataset(dir2+'cm1out_pdata_250m.nc') #***

# # Restricts the timesteps of the data from timesteps0 to 140
# data=data.isel(time=np.arange(0,400+1))
# # # parcel=parcel.isel(time=np.arange(0,400+1))
# res='250m'

In [ ]:
import sys
dir2='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
path=dir2+'../Functions/'
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions


# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(PlottingFunctions, inspect.isfunction)]
# functions

In [ ]:
# Reading Back Data Later
##############
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Binary_Array/'
with h5py.File(dir2+f'lagrangian_binary_array_{res}_{Np_str}.h5', 'r') as f:
    # Load the dataset by its name
    A_g = f['A_g'][:]
    A_c = f['A_c'][:]

    # W = f['W'][:]
    # QCQI = f['QCQI'][:]
    Z = f['Z'][:]
    Y = f['Y'][:]
    X = f['X'][:]

# #Making Time Matrix
# rows, cols = A.shape[0], A.shape[1]
# T = np.arange(rows).reshape(-1, 1) * np.ones((1, cols), dtype=int)

In [ ]:
#job array setup #UNCOMMENT IF USING JOB_ARRAY
job_array=False
if job_array==True:
    num_jobs=60 #how many total jobs are being run? i.e. array=1-100 ==> num_jobs=100 #***
    job_id = int(os.environ.get('SLURM_ARRAY_TASK_ID', 0)) #this is the current SBATCH job id
    if job_id==0: job_id=1
        
    num_parcels=len(data['time']) #total num of variables
    job_range = num_parcels//num_jobs #number of parcels per job 
    start_job = (job_id - 1) * job_range
    end_job = start_job + job_range
    if job_id==num_jobs: end_job=num_parcels-1
    
    
    #FOR ENTRAINMENT THE PREVIOUS TIMESTEP IS ALWAYS NEEDED, SO SLICED WILL CAUSE WRAP AROUND DATA
    #INSTEAD NP.WHERE RESULTS MUST BE SUBSETTED
    # data=data.isel(time=slice(start_job,end_job))
    # parcel=parcel.isel(time=slice(start_job,end_job))
    
    # #SLICING LAGRANGIAN BINARY ARRAY
    # A_g=A_g[slice(start_job,end_job)]
    # A_c=A_c[slice(start_job,end_job)]
    # Z=Z[slice(start_job,end_job)]
    # Y=Y[slice(start_job,end_job)]
    # X=X[slice(start_job,end_job)]

In [ ]:
###########################################################################################################################################################################

In [ ]:
#OLD: FINDS TRIPLETS (101==>111) and (010==>000)
###### (amount of time outside of cloud to count as entrainment)
mins_thresh=5 #5 minutes 
######
def PreProcessing(p,type,updraft_type):
    

    if updraft_type=='general':
        A=A_g
    elif updraft_type=='cloudy':
        A=A_c
    
    B = A[:,p]

    ################################################################
    if type=='e': #TESTING
        B=np.array([1,1,0,0,1,1,1,0,1,0,0,1,1,0,1,1]) #TESTING
        # B=np.array([1,1,0,1,0,1,0,0,1,0,1,0])
        B=np.array([1,1,0,1,0,1,1])

    elif type=='d': #TESTING
        B=np.array([0,0,1,1,0,0,0,1,0,1,1,0,0,1,0,0]) #TESTING
        # B=np.array([1,1,0,1,0,1,0,0,1,0,1,0])
        B=np.array([1,1,0,1,0,1,1])
    print(f'{B}\n') #TESTING
    ################################################################
    
    T=np.arange(len(B))
    
    if np.any(B)==True:
        if type=='e':
            C=B.copy()
        elif type=='d':
            C=1-B.copy()
            

        # Find the changes in the array
        changes = np.diff(np.concatenate(([1], C, [1])))  # Add 0s to detect edges
        print(changes) #TESTING
            
        start_ind = np.where(changes == -1)[0]  # Start of sequences
        end_ind = np.where(changes == 1)[0]  # End of sequences
        
        # Calculate the lengths of sequences
        lengths = end_ind - start_ind

        sequences = [np.arange(start,end) for start, end, length in zip(start_ind, end_ind, lengths) if length >= 1] #only records en/detrainment time
        # print(sequences) #TESTING
        
        # sequences = [(start, *range(start + 1, end+1)) for start, end, length in zip(start_ind, end_ind, lengths) if length >= 1]
        lens=[(end-start) for start, end, length in zip(start_ind, end_ind, lengths) if length >= 1] #residence times
        # print(lens) #TESTING

        #find which lengths are <= 5 minutes, 10 minutes, etc and fill with 1s (entrainment), or 0s (detrainment)
        mins=((data['time'][1]-data['time'][0])/1e9/60).item()
        remove_inds=np.where(np.array(lens)*mins<=mins_thresh) #fills 0 runs that are less than 'mins_thresh' minutes in length
        # print(remove_inds) #TESTING
        run_inds=[sequences[i] for i in remove_inds[0]];
        if len(run_inds) !=0:
            run_inds=np.concatenate(run_inds)
            # print(run_inds) #TESTING
            if type=='e':
                B[run_inds]=1
            if type=='d':
                B[run_inds]=0
    return B

print('entrainment preprocessing')
p=42483; out=PreProcessing(p,type='e',updraft_type='cloudy') #TESTING
print(f'{out}\n')

print('detrainment preprocessing')
p=42483; out=PreProcessing(p,type='d',updraft_type='cloudy') #TESTING
print(f'{out}\n')

In [ ]:
#RUNNING
A_g_Processed_e=A_g.copy(); A_g_Processed_d=A_g.copy()
A_c_Processed_e=A_c.copy(); A_c_Processed_d=A_c.copy()
print('processing general entrainment + detrainment')
for p in np.arange(len(parcel['xh'])-1):
    if np.mod(p,25000)==0: print(f"{p}/{len(parcel['xh'])}")
    out=PreProcessing(p,type='e',updraft_type='general'); A_g_Processed_e[:,p]=out
    out=PreProcessing(p,type='d',updraft_type='general');A_g_Processed_d[:,p]=out
print('processing cloudy entrainment + detrainment')
for p in np.arange(len(parcel['xh'])-1):
    if np.mod(p,25000)==0: print(f"{p}/{len(parcel['xh'])}")
    out=PreProcessing(p,type='e',updraft_type='cloudy'); A_c_Processed_e[:,p]=out
    out=PreProcessing(p,type='d',updraft_type='cloudy');A_c_Processed_d[:,p]=out

#SAVING
dir3=dir+f'Project_Algorithms/Entrainment/processed_binary_arrays_'+str(mins_thresh)+f'mins_{res}_{Np_str}.h5'
with h5py.File(dir3, 'w') as h5file:
    h5file.create_dataset('A_g_Processed_e', data=A_g_Processed_e)
    h5file.create_dataset('A_g_Processed_d', data=A_g_Processed_d)
    h5file.create_dataset('A_c_Processed_e', data=A_c_Processed_e)
    h5file.create_dataset('A_c_Processed_d', data=A_c_Processed_d)
print('done')
#about 50 minutes for 1 million parcels (general+cloudy)